In [2]:
import zipfile
import os
import pandas as pd

metar = "master_metar.xlsx"
drishti = "Processed_RWY27_30min.zip"
extract_path = "extracted"
final = "master_RWY27.xlsx"

df_main = pd.read_excel(metar)
df_main["Date"] = pd.to_datetime(df_main["Date"])
print("Main file loaded.")

with zipfile.ZipFile(drishti, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("Drishti ZIP extracted.")

dataframes = []

for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.endswith(".xlsx"):
            file_path = os.path.join(root, file)

            try:
                df = pd.read_excel(file_path)

                filename = os.path.splitext(file)[0]
                base_date = pd.to_datetime(filename, format="%d.%m.%Y")
                df["Time"] = pd.to_datetime(df["Time"], errors="coerce").dt.time
                df["DateTime"] = df["Time"].apply(
                    lambda t: pd.Timestamp.combine(base_date.date(), t)
                    if pd.notnull(t) else pd.NaT
                )

                df = df[["DateTime", "RVR"]]
                df = df.rename(columns={"RVR": "Drishti"})
                df = df.rename(columns={"DateTime": "Date"})
                dataframes.append(df)

            except Exception as e:
                print(f"Error in {file}: {e}")

df_drishti = pd.concat(dataframes, ignore_index=True)

print("Drishti data combined.")

final_df = pd.merge(df_main, df_drishti, on="Date", how="inner")

final_df = final_df.sort_values("Date")

final_df.to_excel(final, index=False)

print("Final merged Excel created successfully!")
print("Saved at:", final)

Main file loaded.
Drishti ZIP extracted.


/tmp/ipykernel_164/3602116490.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Time"] = pd.to_datetime(df["Time"], errors="coerce").dt.time
/tmp/ipykernel_164/3602116490.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Time"] = pd.to_datetime(df["Time"], errors="coerce").dt.time
/tmp/ipykernel_164/3602116490.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Time"] = pd.to_datetime(df["Time"], errors="coerce").dt.time
/tmp/ipykernel_164/3602116490.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `

Drishti data combined.
Final merged Excel created successfully!
Saved at: master_RWY27.xlsx
